# 🔧 Notebook 2 — Fine-tune Whisper on Comorian
### Train Whisper to understand Shikomori using your corrected audio+text pairs

---
## 📋 Prerequisites
- You need **10-50+ hours** of corrected audio+text pairs from `01_transcribe.ipynb`
- Run on **Kaggle** (30h/week free GPU) or **Colab** (T4 GPU)

## 📋 Steps
| Step | What happens |
|------|--------------|
| 1 | Install packages |
| 2 | Load corrected dataset from Drive |
| 3 | Prepare dataset for training |
| 4 | Load Whisper base model |
| 5 | Train (fine-tune) |
| 6 | Evaluate & save model |

> 💡 **GPU Required:** `Runtime` → `Change runtime type` → `T4 GPU`

## ✅ Step 1: Install Packages

In [ ]:
print('📦 Installing packages...')
!pip install transformers datasets accelerate evaluate jiwer tensorboard -q
!pip install pydub soundfile librosa -q
!apt-get install -y ffmpeg -q 2>/dev/null
print('✅ All packages installed!')

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = round(torch.cuda.get_device_properties(0).total_mem / 1024**3, 1)
    print(f'✅ GPU ready: {gpu_name} ({gpu_mem} GB)')
else:
    print('⚠️  No GPU — fine-tuning requires a GPU!')

print(f'🔧 Device: {device}')

## ✅ Step 2: Load Corrected Dataset from Drive

Upload your corrected audio+text pairs. Expected structure:
```
SHIKOMORI_dataset/
├── corrected/
│   ├── audio/       ← .wav files (16kHz mono)
│   └── metadata.csv ← filename,text pairs
```

In [ ]:
from google.colab import auth, userdata
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import os, io

auth.authenticate_user()
drive_service = build('drive', 'v3')

DATASET_ID = userdata.get('SHIKOMORI_DATASET_ID')
print(f'✅ Connected to Drive\n')

# Download corrected dataset
print('📂 Looking for corrected/ folder...')
print('   Expected: SHIKOMORI_dataset/corrected/audio/ + metadata.csv')
print()
print('⚠️  If you haven\'t created this yet:')
print('   1. Create folder: SHIKOMORI_dataset/corrected/audio/')
print('   2. Put your corrected .wav files in audio/')
print('   3. Create metadata.csv with columns: filename,text')
print('      Example:')
print('      recording_001.wav,Salamualaikum warahmatullahi wabarakatuh')
print('      recording_002.wav,Habar Zaho wa jeje')

## ✅ Step 3: Prepare Dataset for Training

Converts your audio+text pairs into the HuggingFace format Whisper expects.

In [ ]:
from datasets import Dataset, DatasetDict, Audio
import pandas as pd
import soundfile as sf
import librosa

# Load metadata
CORRECTED_DIR = '/content/corrected'
os.makedirs(CORRECTED_DIR, exist_ok=True)

print('📋 To prepare your dataset:')
print()
print('   Option A — Upload manually:')
print('   1. Upload .wav files to /content/corrected/audio/')
print('   2. Upload metadata.csv to /content/corrected/')
print()
print('   Option B — From Google Drive (once you have corrected data):')
print('   Mount Drive and copy corrected/ folder')
print()
print('   metadata.csv format:')
print('   ┌──────────────────────┬────────────────────────────────────────┐')
print('   │ filename             │ text                                   │')
print('   ├──────────────────────┼────────────────────────────────────────┤')
print('   │ recording_001.wav    │ Salamualaikum warahmatullahi           │')
print('   │ recording_002.wav    │ Habar Zaho wa jeje wa mirongolo       │')
print('   └──────────────────────┴────────────────────────────────────────┘')
print()

metadata_path = f'{CORRECTED_DIR}/metadata.csv'
if os.path.exists(metadata_path):
    df = pd.read_csv(metadata_path)
    print(f'✅ Loaded metadata: {len(df)} samples')
    print(f'   Columns: {list(df.columns)}')
    print(f'   Preview:')
    print(df.head())

    # Build HuggingFace dataset
    audio_paths = [f'{CORRECTED_DIR}/audio/{fn}' for fn in df['filename']]
    dataset = Dataset.from_dict({
        'audio': audio_paths,
        'text': df['text'].tolist(),
    }).cast_column('audio', Audio(sampling_rate=16000))

    # Split 90/10
    split = dataset.train_test_split(test_size=0.1, seed=42)
    print(f'\n📊 Dataset split:')
    print(f'   Train: {len(split["train"])} samples')
    print(f'   Test:  {len(split["test"])} samples')
else:
    print(f'⚠️  No metadata.csv found at {metadata_path}')
    print(f'   Upload your corrected data first (see instructions above)')
    split = None

## ✅ Step 4: Load Whisper Model for Fine-tuning

We fine-tune `openai/whisper-small` (244M params) — best balance of quality vs training speed for limited GPU.

In [ ]:
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate

MODEL_ID = 'openai/whisper-small'

print(f'🔄 Loading {MODEL_ID}...')
processor = WhisperProcessor.from_pretrained(MODEL_ID, language='sw', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

# Set language and task tokens
model.generation_config.language = 'sw'
model.generation_config.task = 'transcribe'
model.generation_config.forced_decoder_ids = None

print(f'✅ {MODEL_ID} loaded!')
print(f'   Parameters: 244M')
print(f'   Language: Swahili (sw) — will adapt to Comorian through fine-tuning')

## ✅ Step 5: Prepare Data Collator & Metrics

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

if split is not None:
    print('🔄 Processing dataset...')
    split = split.map(prepare_dataset, remove_columns=split.column_names["train"], num_proc=1)
    print('✅ Dataset processed!')
else:
    print('⚠️  No dataset loaded — upload corrected data in Step 2')

## ✅ Step 6: Train!

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-comorian",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=500,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

if split is not None:
    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=split["train"],
        eval_dataset=split["test"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        tokenizer=processor.feature_extractor,
    )

    print('🚀 Starting fine-tuning...')
    print(f'   Batch size: {training_args.per_device_train_batch_size}')
    print(f'   Steps: {training_args.max_steps}')
    print(f'   Learning rate: {training_args.learning_rate}')
    print()

    trainer.train()

    print('\n🎉 Training complete!')
else:
    print('⚠️  No dataset — upload corrected data first')

## ✅ Step 7: Evaluate & Save

In [ ]:
if split is not None:
    print('📊 Evaluating on test set...')
    results = trainer.evaluate()
    print(f'\n✅ Results:')
    print(f'   WER: {results["eval_wer"]:.1f}%')
    print(f'   Loss: {results["eval_loss"]:.4f}')

    # Save model
    model.save_pretrained('./whisper-comorian-final')
    processor.save_pretrained('./whisper-comorian-final')
    print(f'\n💾 Model saved to ./whisper-comorian-final/')

    print(f'\n📤 To upload to HuggingFace Hub:')
    print(f'   1. Create account at huggingface.co')
    print(f'   2. Run: !huggingface-cli login')
    print(f'   3. Run: trainer.push_to_hub()')
else:
    print('⚠️  No training done — upload corrected data first')

---
## ✅ Done! Next Steps

1. **Use your fine-tuned model** in `01_transcribe.ipynb` by changing `MODEL_SIZE` to your model path
2. **Transcribe more audio** with the improved model → correct → retrain (snowball!)
3. When you have **100+ hours** → train a larger model or move to **`03_translate.ipynb`**
4. **Publish** your model on HuggingFace Hub to help other Comorian speakers

---
*Built for the Comorian (Shikomori) Language Preservation Project* 🇰🇲